# H-005 · Size & Value Feature Suite

Factor test for **H-005** (equities): whether size / valuation levels, valuation rate-of-change, size momentum, and value–momentum interaction features carry cross-sectional predictive power for forward returns at Alphalens `periods=(1, 5, 21)` (primary narrative **5d**).

- **Idea** — Daily `market_cap` / `pe` / `pb` via `add_size_value_factors` (auto-fetch unless `size_value_data_exists=True`); screen a balanced window / momentum grid on research IS only.
- **Claim** — Value levels, valuation RoC, and value–momentum geometry improve next-week IC beyond size alone.
- **Why it might work** — Classic value (book / earnings yield); “getting cheaper/richer” (Δlog valuation) often predicts better at 5–21d than static levels; value and momentum are negatively correlated so interaction / distance / residual features can lift spreads.
- **Data** — Daily OHLCV long panel (`s1_factor_panel_train.parquet`); SEC size/value attaches inside `add_size_value_factors`.

## Features (8 store callers)

| Store caller | Output column(s) | `normalize` |
|---|---|---|
| `add_size_value_factors` | `book_yield` | True (CS pct-rank) |
| `add_size_value_factors` | `earnings_yield` | True |
| `add_size_value_factors` | `log_mcap` | True |
| `add_size_value_factors` | `val_roc_{metric}` [`_{W}`] | **none** (never CS-ranked) |
| `add_size_value_factors` | `size_mom` [`_{W}`] | **none** (never CS-ranked) |
| `add_size_value_factors` | `val_mom_interact` | **none** (never CS-ranked) |
| `add_size_value_factors` | `val_mom_dist` | **none** (never CS-ranked) |
| `add_size_value_factors` | `val_mom_resid` | **none** (never CS-ranked) |

**Normalize policy:** CS pct-rank only for level features (yields, log mcap). RoC / size-mom / value–momentum features are already Δlog, rank-space, or z-scored — no second CS rank.

## Size/value feature parquet cache

| Path | Role |
|------|------|
| `01_data/data_files/s1_equities/s1_factor_panel_train.parquet` | Research IS OHLCV (cold path only) |
| `01_data/data_files/s1_equities/s1_h005_sv_panel.parquet` | Cached panel with H-005 columns |

**Save gate:** on a cold build, after features are built, write the SV parquet **before** any Alphalens screen / tear sheet.

**Invalidate** by deleting the SV parquet or setting `FORCE_REBUILD = True` when changing grid constants or store code.

## Sample discipline

- Do **not** use `s1_factor_panel_full.parquet` for window keep/kill.
- Overlapping 5d / 21d labels warrant purge/embargo in later walk-forward; this notebook screens IC on research IS only.
- Variant count = number of H-005 factor columns screened (see `EXPECTED_N_FACTORS`).

Evaluation uses the S1 **trade-date** panel: Alphalens pivots `open` (no `shift(-1)`); labels are open-to-open. Prior close-to-close ICs are not comparable.


## 0. Imports & Config

Resolve the repo root; configure the parameter grid, SV-cache paths, and Alphalens periods. Set `FORCE_REBUILD = True` to ignore an existing SV parquet and rebuild from the train IS.


In [1]:
from __future__ import annotations

import itertools
import os
import re
import sys
import time

# Jupyter cwd is often this notebook's folder, not the repo root; walk up until we find 01_data/ingestion.
ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import alphalens as al
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from matplotlib.backends.backend_pdf import PdfPages

from data.processing.s1_feature_store import add_size_value_factors

TRAIN_PANEL_PATH = os.path.join(
    ROOT, "01_data", "data_files", "s1_equities", "s1_factor_panel_train.parquet"
)
SV_PANEL_PATH = os.path.join(
    ROOT, "01_data", "data_files", "s1_equities", "s1_h005_sv_panel.parquet"
)
TEARSHEET_DIR = os.path.join(
    ROOT, "02_research", "notebooks", "s1_equities", "factor_tests", "tearsheets"
)

# --- Parameter grid (edit these lists) ---
WINDOWS = [21, 63, 126, 252]           # val_roc / size_mom windows
MOM_LOOKBACKS = [126, 252]             # value-momentum lookbacks
MOM_SKIPS = [10, 21, 42]               # value-momentum skips
REGRESSION_WINDOWS = [126, 252]        # val_mom_resid OLS windows
VAL_METRICS = ["pe", "pb"]             # valuation_roc metrics

EXPECTED_N_FACTORS = (
    3  # book_yield, earnings_yield, log_mcap
    + len(VAL_METRICS) * len(WINDOWS)  # val_roc_{metric}_{W}
    + len(WINDOWS)  # size_mom_{W}
    + 2 * len(MOM_LOOKBACKS) * len(MOM_SKIPS)  # interact + dist
    + len(REGRESSION_WINDOWS) * len(MOM_LOOKBACKS) * len(MOM_SKIPS)  # resid
)

# --- Fixed for this notebook ---
FORCE_REBUILD = False
FFILL_LIMIT = 5  # business days; avoid stale fundamentals
PERIODS = (1, 5, 21)  # primary narrative = 5d
QUANTILES = 5
MAX_LOSS = 0.35

H005_PREFIXES = (
    "book_yield",
    "earnings_yield",
    "log_mcap",
    "val_roc_",
    "size_mom",
    "val_mom_interact",
    "val_mom_dist",
    "val_mom_resid",
)

print(f"ROOT={ROOT}")
print(f"EXPECTED_N_FACTORS={EXPECTED_N_FACTORS}")
print(f"FORCE_REBUILD={FORCE_REBUILD}")


ROOT=c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio
EXPECTED_N_FACTORS=39
FORCE_REBUILD=False


## 1. Data Loading

### Size/value parquet cache contract

1. If `FORCE_REBUILD` is False **and** `s1_h005_sv_panel.parquet` exists → **CACHE HIT**: load it into `panel` (features already present).
2. Otherwise → **CACHE MISS / cold build**: load `s1_factor_panel_train.parquet`; first `add_size_value_factors(..., size_value_data_exists=False)` fetches SEC fields (later calls use `True`).
3. On a warm load, validate H-005 column count. If the cache looks stale (wrong count), fall through to a cold build.

Do **not** apply another 70/30 split here — the train parquet is already research IS.


In [2]:
def _h005_factor_cols(frame: pd.DataFrame) -> list[str]:
    cols = []
    for c in frame.columns:
        if c in ("book_yield", "earnings_yield", "log_mcap"):
            cols.append(c)
        elif c == "size_mom" or c.startswith("size_mom_"):
            cols.append(c)
        elif c.startswith("val_roc_"):
            cols.append(c)
        elif c.startswith("val_mom_interact"):
            cols.append(c)
        elif c.startswith("val_mom_dist"):
            cols.append(c)
        elif c.startswith("val_mom_resid"):
            cols.append(c)
    return cols


use_sv_cache = (
    (not FORCE_REBUILD)
    and os.path.isfile(SV_PANEL_PATH)
)

if use_sv_cache:
    panel = pd.read_parquet(SV_PANEL_PATH)
    panel["date"] = pd.to_datetime(panel["date"])
    FACTOR_COLS = _h005_factor_cols(panel)
    if len(FACTOR_COLS) != EXPECTED_N_FACTORS:
        print(
            f"CACHE STALE: found {len(FACTOR_COLS)} H-005 cols "
            f"(expected {EXPECTED_N_FACTORS}) - falling back to cold build"
        )
        use_sv_cache = False
    else:
        n_tickers = panel["ticker"].nunique()
        n_dates = panel["date"].nunique()
        print(
            f"CACHE HIT: {SV_PANEL_PATH}\n"
            f"rows={len(panel):,}  tickers={n_tickers}  dates={n_dates:,}  "
            f"factor cols={len(FACTOR_COLS)}  "
            f"[{panel['date'].min().date()} -> {panel['date'].max().date()}]"
        )

if not use_sv_cache:
    panel = pd.read_parquet(TRAIN_PANEL_PATH)
    panel["date"] = pd.to_datetime(panel["date"])
    panel["ticker"] = panel["ticker"].astype(str).str.strip().str.upper()
    tickers = sorted(panel["ticker"].unique().tolist())
    start = panel["date"].min().date()
    end = panel["date"].max().date()
    print(
        f"CACHE MISS: loaded {TRAIN_PANEL_PATH}\n"
        f"rows={len(panel):,}  tickers={len(tickers)}  "
        f"[{start} -> {end}]  "
        f"(SEC size/value attaches inside add_size_value_factors)"
    )
    FACTOR_COLS = []

panel.head()


CACHE MISS: loaded c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\01_data\data_files\s1_equities\s1_factor_panel_train.parquet
rows=289,381  tickers=100  [2010-01-05 -> 2021-08-03]  (will fetch size/value + build features)


No SEC CIK for ticker AET; leaving fundamentals NaN
No SEC CIK for ticker ESRX; leaving fundamentals NaN
No SEC CIK for ticker TWX; leaving fundamentals NaN


SV merge: market_cap non-null=86.8%  pe=66.9%  pb=77.3%


,date,ticker,open,high,low,close,volume,feature_date,fwd_ret_1,fwd_ret_5,fwd_ret_21,shares_outstanding,book_equity,eps_ttm,market_cap,pe,pb
0,2010-01-05,AAPL,6.424143,6.421146,6.357683,6.406478,493729600.0,2010-01-04,-0.001025,-0.025210,-0.083271,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-01-06,AAPL,6.417558,6.453779,6.383729,6.417557,601904800.0,2010-01-05,-0.012268,-0.030367,-0.101456,900678473.0,2.783200e+10,NaN,5.780155e+09,NaN,0.207680
2,2010-01-07,AAPL,6.338825,6.443003,6.308892,6.315478,552160000.0,2010-01-06,-0.006848,-0.007745,-0.075844,900678473.0,2.783200e+10,NaN,5.688215e+09,NaN,0.204377
3,2010-01-08,AAPL,6.295419,6.346309,6.258000,6.303801,477131200.0,2010-01-07,0.011888,0.002996,-0.066001,900678473.0,2.783200e+10,NaN,5.677697e+09,NaN,0.203999
4,2010-01-11,AAPL,6.370258,6.346310,6.258300,6.345711,447610800.0,2010-01-08,-0.016965,-0.021005,-0.079464,900678473.0,2.783200e+10,NaN,5.715445e+09,NaN,0.205355


## 2. Data Cleaning & Engineering

Limited forward-fill of size/value columns runs in §3 immediately after the first
`add_size_value_factors` fetch (`FFILL_LIMIT` business days). No floor / winsorize
in library code — apply any extra cleaning here only.


In [3]:
# SEC size/value ffill is applied in §3 after the first store fetch.
print("skip §2 ffill (handled after first add_size_value_factors on cold path)")


NaN coverage after limited ffill (cold path only applies ffill):
  market_cap: non-null=86.9%
  pe: non-null=68.8%
  pb: non-null=77.4%
  shares_outstanding: non-null=86.9%
  book_equity: non-null=86.4%
  eps_ttm: non-null=73.6%


## 3. Modeling / Signal Construction

Build all H-005 columns on the cold path. Level features use store defaults (`normalize=True`). Windowed / value–momentum features never CS-rank.

Value–momentum grids call raw panel helpers with explicit column names (store callers take single lookback/skip ints).

**Save gate (cold path):** after build, write `s1_h005_sv_panel.parquet` **before** Alphalens.


In [4]:
t0 = time.perf_counter()

if use_sv_cache:
    print("Skipping §3 rebuild - using cached H-005 columns")
    FACTOR_COLS = _h005_factor_cols(panel)
else:
    # Level features; first call fetches SEC size/value
    panel = add_size_value_factors(
        panel,
        feature_subset=["book_yield", "earnings_yield", "log_mcap"],
        size_value_data_exists=False,
    )
    sv_cols = [
        c
        for c in (
            "market_cap",
            "pe",
            "pb",
            "shares_outstanding",
            "book_equity",
            "eps_ttm",
        )
        if c in panel.columns
    ]
    panel = panel.sort_values(["ticker", "date"]).copy()
    panel[sv_cols] = panel.groupby("ticker", sort=False)[sv_cols].ffill(
        limit=FFILL_LIMIT
    )
    print(
        f"SV after fetch+ffill: market_cap non-null="
        f"{panel['market_cap'].notna().mean():.1%}  "
        f"pe={panel['pe'].notna().mean():.1%}  pb={panel['pb'].notna().mean():.1%}"
    )

    panel = add_size_value_factors(
        panel,
        feature_subset=[f"val_roc_{m}" for m in VAL_METRICS] + ["size_mom"],
        window=WINDOWS,
        size_value_data_exists=True,
    )

    for L, S in itertools.product(MOM_LOOKBACKS, MOM_SKIPS):
        if L <= S:
            raise ValueError(f"mom_lookback must be > mom_skip, got L={L}, S={S}")
        tmp = add_size_value_factors(
            panel,
            feature_subset=["val_mom_interact", "val_mom_dist"],
            mom_lookback=L,
            mom_skip=S,
            size_value_data_exists=True,
        )
        panel[f"val_mom_interact_{L}_{S}"] = tmp["val_mom_interact"]
        panel[f"val_mom_dist_{L}_{S}"] = tmp["val_mom_dist"]

    for RW, L, S in itertools.product(REGRESSION_WINDOWS, MOM_LOOKBACKS, MOM_SKIPS):
        if L <= S:
            raise ValueError(f"mom_lookback must be > mom_skip, got L={L}, S={S}")
        tmp = add_size_value_factors(
            panel,
            feature_subset=["val_mom_resid"],
            regression_window=RW,
            mom_lookback=L,
            mom_skip=S,
            size_value_data_exists=True,
        )
        panel[f"val_mom_resid_{RW}_{L}_{S}"] = tmp["val_mom_resid"]

    FACTOR_COLS = _h005_factor_cols(panel)
    assert len(FACTOR_COLS) == EXPECTED_N_FACTORS, (
        f"expected {EXPECTED_N_FACTORS} factor cols, got {len(FACTOR_COLS)}"
    )

    os.makedirs(os.path.dirname(SV_PANEL_PATH), exist_ok=True)
    panel.to_parquet(SV_PANEL_PATH, index=False)
    print(f"Wrote {SV_PANEL_PATH}")

elapsed = time.perf_counter() - t0
print(f"H-005 factor columns: {len(FACTOR_COLS)}  (build/load wall={elapsed:.1f}s)")
assert len(FACTOR_COLS) == EXPECTED_N_FACTORS, (
    f"expected {EXPECTED_N_FACTORS} factor cols, got {len(FACTOR_COLS)}"
)
print("Factor columns:")
for c in sorted(FACTOR_COLS):
    print(f"  {c}")


c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, me

Wrote c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\01_data\data_files\s1_equities\s1_h005_sv_panel.parquet
H-005 factor columns: 39  (build/load wall=319.0s)
Factor columns:
  book_yield
  earnings_yield
  log_mcap
  size_mom_126
  size_mom_21
  size_mom_252
  size_mom_63
  val_mom_dist_126_10
  val_mom_dist_126_21
  val_mom_dist_126_42
  val_mom_dist_252_10
  val_mom_dist_252_21
  val_mom_dist_252_42
  val_mom_interact_126_10
  val_mom_interact_126_21
  val_mom_interact_126_42
  val_mom_interact_252_10
  val_mom_interact_252_21
  val_mom_interact_252_42
  val_mom_resid_126_126_10
  val_mom_resid_126_126_21
  val_mom_resid_126_126_42
  val_mom_resid_126_252_10
  val_mom_resid_126_252_21
  val_mom_resid_126_252_42
  val_mom_resid_252_126_10
  val_mom_resid_252_126_21
  val_mom_resid_252_126_42
  val_mom_resid_252_252_10
  val_mom_resid_252_252_21
  val_mom_resid_252_252_42
  val_roc_pb_126
  val_roc_pb_21
  val_roc_pb_252
  val_roc_pb_63
  val_roc_pe_126
 

## 4. Evaluation

Screen every H-005 column at `periods=(1, 5, 21)` with `quantiles=5`. Primary sort key: **`ic_5d`**. Decode column parameters with `parse_sv_factor_name`.

Full tear sheets are **PDF-only** (no inline display) — edit `TEAR_FACTORS` after reviewing §4.1.


In [5]:
def to_alphalens_prices(panel: pd.DataFrame) -> pd.DataFrame:
    """Wide open matrix for Alphalens (trade-date panel; entry at open)."""
    prices = panel.pivot(index="date", columns="ticker", values="open")
    prices.index = pd.to_datetime(prices.index)
    return prices.sort_index()


def to_alphalens_factor(panel: pd.DataFrame, col: str) -> pd.Series:
    """MultiIndex (date, ticker) factor series for Alphalens."""
    factor = panel.set_index(["date", "ticker"])[col].dropna()
    factor.index = factor.index.set_levels(
        pd.to_datetime(factor.index.levels[0]), level=0
    )
    return factor.sort_index()


def _period_label(period_index: pd.Index, period: int, position: int):
    """Match Alphalens period label ('1D', '5D', ...) or fall back by position."""
    for c in (f"{period}D", f"{period}d", period, str(period)):
        if c in period_index:
            return c
    return period_index[position]


def parse_sv_factor_name(col: str) -> dict | None:
    """Decode an H-005 factor column into family / window metadata."""
    if col in ("book_yield", "earnings_yield", "log_mcap"):
        return {"feature": col, "family": col}

    m = re.fullmatch(r"val_roc_(pe|pb)_(\d+)", col)
    if m:
        return {
            "feature": f"val_roc_{m.group(1)}",
            "family": f"val_roc_{m.group(1)}",
            "metric": m.group(1),
            "window": int(m.group(2)),
        }
    m = re.fullmatch(r"val_roc_(pe|pb)", col)
    if m:
        return {
            "feature": f"val_roc_{m.group(1)}",
            "family": f"val_roc_{m.group(1)}",
            "metric": m.group(1),
        }

    m = re.fullmatch(r"size_mom_(\d+)", col)
    if m:
        return {"feature": "size_mom", "family": "size_mom", "window": int(m.group(1))}
    if col == "size_mom":
        return {"feature": "size_mom", "family": "size_mom"}

    m = re.fullmatch(r"val_mom_interact_(\d+)_(\d+)", col)
    if m:
        return {
            "feature": "val_mom_interact",
            "family": "val_mom_interact",
            "lookback": int(m.group(1)),
            "skip": int(m.group(2)),
        }
    if col == "val_mom_interact":
        return {"feature": "val_mom_interact", "family": "val_mom_interact"}

    m = re.fullmatch(r"val_mom_dist_(\d+)_(\d+)", col)
    if m:
        return {
            "feature": "val_mom_dist",
            "family": "val_mom_dist",
            "lookback": int(m.group(1)),
            "skip": int(m.group(2)),
        }
    if col == "val_mom_dist":
        return {"feature": "val_mom_dist", "family": "val_mom_dist"}

    m = re.fullmatch(r"val_mom_resid_(\d+)_(\d+)_(\d+)", col)
    if m:
        return {
            "feature": "val_mom_resid",
            "family": "val_mom_resid",
            "reg_window": int(m.group(1)),
            "lookback": int(m.group(2)),
            "skip": int(m.group(3)),
        }
    if col == "val_mom_resid":
        return {"feature": "val_mom_resid", "family": "val_mom_resid"}

    return None


def factor_screen_metrics(
    factor: pd.Series,
    prices: pd.DataFrame,
    *,
    periods: tuple[int, ...] = PERIODS,
    quantiles: int = QUANTILES,
    max_loss: float = MAX_LOSS,
) -> dict:
    """Mean IC and Q5-Q1 mean return spread for each forward period."""
    factor_data = al.utils.get_clean_factor_and_forward_returns(
        factor=factor,
        prices=prices,
        quantiles=quantiles,
        periods=periods,
        max_loss=max_loss,
    )
    mean_ic = al.performance.mean_information_coefficient(factor_data)
    mean_ret, _ = al.performance.mean_return_by_quantile(factor_data, demeaned=True)

    row = {}
    for i, p in enumerate(periods):
        ic_key = _period_label(mean_ic.index, p, i)
        ret_key = _period_label(mean_ret.columns, p, i)
        row[f"ic_{p}d"] = float(mean_ic.loc[ic_key])
        q_hi, q_lo = mean_ret.index.max(), mean_ret.index.min()
        row[f"spread_{p}d"] = float(
            mean_ret.loc[q_hi, ret_key] - mean_ret.loc[q_lo, ret_key]
        )
    return row


def run_full_tear(
    panel: pd.DataFrame,
    factor_col: str,
    prices: pd.DataFrame,
    *,
    periods: tuple[int, ...] = PERIODS,
    quantiles: int = QUANTILES,
    max_loss: float = MAX_LOSS,
    tearsheet_dir: str = TEARSHEET_DIR,
):
    """Build factor_data, run Alphalens full tear, save figs to multi-page PDF.

    PDF-only: Agg backend + patched ``plt.show`` so figures are not displayed
    inline (many H-005 variants).
    """
    if factor_col not in panel.columns:
        raise ValueError(
            f"{factor_col!r} not in panel - pick a screened column "
            f"(available: {FACTOR_COLS})"
        )
    plt.close("all")
    plt.ioff()
    factor_data = al.utils.get_clean_factor_and_forward_returns(
        factor=to_alphalens_factor(panel, factor_col),
        prices=prices,
        quantiles=quantiles,
        periods=periods,
        max_loss=max_loss,
    )

    os.makedirs(tearsheet_dir, exist_ok=True)
    out_path = os.path.join(tearsheet_dir, f"H-005_{factor_col}.pdf")
    pdf = PdfPages(out_path)
    n_pages = 0
    _original_show = plt.show

    def _show_and_savefig(*args, **kwargs):
        nonlocal n_pages
        for num in list(plt.get_fignums()):
            fig = plt.figure(num)
            if fig.axes:
                pdf.savefig(fig, bbox_inches="tight")
                n_pages += 1
        plt.close("all")

    plt.show = _show_and_savefig
    try:
        al.tears.create_full_tear_sheet(factor_data, long_short=True)
        _show_and_savefig()
    finally:
        plt.show = _original_show
        pdf.close()
        plt.close("all")

    print(f"Wrote {out_path} ({n_pages} pages)")
    return factor_data


### 4.1 Window screen summary

Full IC / spread table sorted by `ic_5d`, plus a **best-by-family** table (one row per feature stem).

**Expected signs (literature / intuition):**
- `book_yield` / `earnings_yield` — positive IC (cheap → long)
- `log_mcap` — often negative IC (size); weak in large-cap sleeves
- `val_roc_*` — context-dependent (“getting cheaper” vs richer)
- `size_mom` — overlaps price momentum; test incremental IC
- `val_mom_interact` — high value × high mom; often positive
- `val_mom_dist` — distance from ideal (1,1); often **negative** IC (closer = better)
- `val_mom_resid` — orthogonal value after mom; sign empirical


In [6]:
t0 = time.perf_counter()
prices = to_alphalens_prices(panel)
rows = []
for col in FACTOR_COLS:
    meta = parse_sv_factor_name(col) or {}
    metrics = factor_screen_metrics(to_alphalens_factor(panel, col), prices)
    rows.append({"factor": col, **meta, **metrics})

summary = (
    pd.DataFrame(rows)
    .sort_values("ic_5d", ascending=False)
    .reset_index(drop=True)
)
print(f"Alphalens screen wall={time.perf_counter() - t0:.1f}s  n={len(summary)}")

summary["feature_family"] = summary["factor"].map(
    lambda c: (parse_sv_factor_name(c) or {}).get("family")
)
best_by_family = (
    summary.sort_values("ic_5d", ascending=False)
    .groupby("feature_family", as_index=False)
    .first()
    .sort_values("ic_5d", ascending=False)
    .reset_index(drop=True)
)

print("\n=== Best by feature family (ic_5d) ===")
print(best_by_family.to_string())

print("\n=== Full screen (sorted by ic_5d) ===")
with pd.option_context("display.max_columns", None, "display.max_rows", None):
    display(summary)


c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change(period)
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change(period)
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any n

Alphalens screen wall=795.1s  n=39

=== Best by feature family (ic_5d) ===
     feature_family                    factor           feature            family     ic_1d  spread_1d     ic_5d  spread_5d    ic_21d  spread_21d metric  window  lookback  skip  reg_window
0          size_mom              size_mom_126          size_mom          size_mom  0.015796   0.000317  0.019825   0.001581  0.030989    0.006789   None   126.0       NaN   NaN         NaN
1        val_roc_pb            val_roc_pb_252        val_roc_pb        val_roc_pb  0.012880   0.000126  0.016187   0.000433  0.023579    0.001563     pb   252.0       NaN   NaN         NaN
2        val_roc_pe            val_roc_pe_252        val_roc_pe        val_roc_pe  0.005508   0.000159  0.013468   0.001131  0.027051    0.005554     pe   252.0       NaN   NaN         NaN
3     val_mom_resid  val_mom_resid_126_252_10     val_mom_resid     val_mom_resid  0.005734   0.000260  0.011463   0.001131  0.010226    0.002617   None     NaN     252.

,factor,feature,family,ic_1d,spread_1d,ic_5d,spread_5d,ic_21d,spread_21d,metric,window,lookback,skip,reg_window,feature_family
0,size_mom_126,size_mom,size_mom,0.015796,0.000317,0.019825,0.001581,0.030989,0.006789,NaN,126.0,NaN,NaN,NaN,size_mom
1,size_mom_252,size_mom,size_mom,0.015694,0.000174,0.017399,0.000838,0.019696,0.002771,NaN,252.0,NaN,NaN,NaN,size_mom
2,val_roc_pb_252,val_roc_pb,val_roc_pb,0.012880,0.000126,0.016187,0.000433,0.023579,0.001563,pb,252.0,NaN,NaN,NaN,val_roc_pb
3,val_roc_pe_252,val_roc_pe,val_roc_pe,0.005508,0.000159,0.013468,0.001131,0.027051,0.005554,pe,252.0,NaN,NaN,NaN,val_roc_pe
4,size_mom_63,size_mom,size_mom,0.010657,0.000154,0.013210,0.000901,0.027400,0.003773,NaN,63.0,NaN,NaN,NaN,size_mom
5,val_mom_resid_126_252_10,val_mom_resid,val_mom_resid,0.005734,0.000260,0.011463,0.001131,0.010226,0.002617,NaN,NaN,252.0,10.0,126.0,val_mom_resid
6,val_mom_resid_126_252_42,val_mom_resid,val_mom_resid,0.004461,0.000242,0.011090,0.000910,0.009989,0.001726,NaN,NaN,252.0,42.0,126.0,val_mom_resid
7,val_roc_pe_126,val_roc_pe,val_roc_pe,0.003347,-0.000014,0.010956,0.000447,0.014821,0.002522,pe,126.0,NaN,NaN,NaN,val_roc_pe
8,val_mom_interact_252_21,val_mom_interact,val_mom_interact,0.009643,0.000071,0.010859,0.000376,0.015579,0.001121,NaN,NaN,252.0,21.0,NaN,val_mom_interact
9,val_mom_interact_252_10,val_mom_interact,val_mom_interact,0.009877,0.000102,0.010569,0.000282,0.014432,0.001349,NaN,NaN,252.0,10.0,NaN,val_mom_interact


### 4.2 Full tear sheets (manual selection)

Edit `TEAR_FACTORS` after reviewing §4.1. Each tear is saved as a multi-page PDF under:

`02_research/notebooks/s1_equities/factor_tests/tearsheets/H-005_{factor_col}.pdf`

**No inline display** (runtime / notebook size). Re-running overwrites the same paths. **Do not** tear all screened columns.


In [7]:
# Edit after reviewing §4.1 (defaults are placeholders)
TEAR_FACTORS = [
    "size_mom_126",
    "val_roc_pb_252",
    "val_roc_pe_252",
    "val_mom_interact_252_21",
    "earnings_yield",
    "log_mcap",
    "val_mom_dist_252_21",
    "val_mom_resid_126_252_10",
]

for tear_col in TEAR_FACTORS:
    print(f"\n===== Tear sheet: {tear_col} =====")
    run_full_tear(panel, tear_col, prices)



===== Tear sheet: size_mom_126 =====


c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change(period)
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change(period)
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any n

Quantiles Statistics


,min,max,mean,std,count,count %
factor_quantile,,,,,,
1,-14.308848,0.207065,-0.253928,0.779892,48290,20.507311
2,-0.670209,0.306511,-0.028023,0.104919,46510,19.751398
3,-0.429465,0.388151,0.053823,0.094335,46488,19.742055
4,-0.296200,0.579021,0.131405,0.097792,46510,19.751398
5,-0.150938,14.004949,0.369738,0.764032,47679,20.247837


Returns Analysis


...

c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\performance.py:118: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  grouper.append(pd.Grouper(freq=by_time))
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:928: UserWarning: Skipping return periods that aren't exact multiples of days.
  warnings.warn(


Turnover Analysis


,1D,5D,21D
Quantile 1 Mean Turnover,0.177,0.348,0.592
Quantile 2 Mean Turnover,0.338,0.555,0.735
Quantile 3 Mean Turnover,0.354,0.568,0.739
Quantile 4 Mean Turnover,0.341,0.564,0.740
Quantile 5 Mean Turnover,0.176,0.348,0.587


,1D,5D,21D
Mean Factor Rank Autocorrelation,0.905,0.734,0.396


Wrote c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\02_research\notebooks\factor_tests\tearsheets\H-005_val_mom_resid_126_252_10.pdf (3 pages)


## 5. Wrap-up / next steps

- Record keep/kill decisions in `02_research/hypothesis_log.md` (H-005 Alphalens summary).
- Holdout: reserved; do not peek at `s1_factor_panel_full.parquet` for keep/kill.
- Tear PDFs under `02_research/notebooks/s1_equities/factor_tests/tearsheets/` named `H-005_{factor_col}.pdf`.
- If cold-run SEC fetch dominates wall time, keep `FORCE_REBUILD = False` after the first successful build.
- If screening wall time is high, trim `WINDOWS` / `MOM_SKIPS` / `REGRESSION_WINDOWS` in §0 and rebuild.
- Next: walk-forward / purged CV before any `feature_spec` freeze.
